# Energy Generation Forecasting with LSTM

This notebook implements a Long Short-Term Memory (LSTM) neural network to forecast daily electricity generation for the **Valparaíso region** of Chile, using historical data from 2016 to 2023.

**Experimental design**

| Split | Period | Days in data | Sequences (Days − 30) | Purpose |
|---|---|---|---|---|
| Training | 2016-01-02 – 2021-12-31 | 2,191 | 2,161 | Model fitting |
| Validation | 2022-01-01 – 2022-12-31 | 365 | 335 | Hyperparameter selection |
| Test | 2023-01-01 – 2023-11-30 | 334 | 304 | Held-out evaluation |

> **Note on test period:** The CNE dataset used here ends on 2023-11-30. The full study covers all of 2023 (365 days → 335 test instances, paper Appendix B.3). The 304-sequence test period used here is the maximal out-of-sample window available in this data extract.

Hyperparameters were selected via Optuna Bayesian optimization (TPE sampler, 50 trials). The final model achieves a test-set **RMSE of 3,626.8 MWh** and **MAPE of 10.98%** on the held-out 2023 data.

> **Setup:** The raw dataset (`data/generacion_total.parquet`) must be placed in the `data/` folder before running Stage 1 (see README for download instructions). Stage 2 (`Modelation_linear_programming.ipynb`) can be run independently using the pre-computed predictions already included in `data/predictions_valparaiso_2023.parquet`.

## 1. Import Libraries

All dependencies are imported upfront. The key libraries are PyTorch (LSTM model), Optuna (hyperparameter optimization), and scikit-learn (preprocessing and metrics).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import optuna

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Suppress Optuna trial-level logs for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Output directories
os.makedirs("figures", exist_ok=True)
os.makedirs("models",  exist_ok=True)
os.makedirs("data",    exist_ok=True)

## 2. Data Loading & Preprocessing

The raw dataset contains hourly generation records per power plant across all Chilean regions. We aggregate to **daily totals** and restrict to the **Valparaíso region** for this case study.

In [ ]:
# Load consolidated national generation dataset
df = pd.read_parquet("data/generacion_total.parquet")

# Filter for Valparaíso region
df = df[df["REGIÓN"] == "Región de Valparaíso"]

# Aggregate all plants to daily total generation
df = df.groupby("FECHA")["TOTAL"].sum().reset_index()

# Restrict to 2016–2023 (sufficient history, avoids structural breaks in early years)
df = df[(df["FECHA"] > "2016-01-01") & (df["FECHA"] < "2023-12-01")]
df.set_index("FECHA", inplace=True)

print(f"Dataset: {len(df)} daily records  |  {df.index.min().date()} → {df.index.max().date()}")
print(f"Generation range: {df['TOTAL'].min():,.0f} – {df['TOTAL'].max():,.0f} MWh/day")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df.index, df["TOTAL"], linewidth=0.9, color="steelblue")
ax.axvspan(pd.Timestamp("2016-01-01"), pd.Timestamp("2022-01-01"),
           alpha=0.07, color="steelblue", label="Training (2016–2021)")
ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2023-01-01"),
           alpha=0.07, color="darkorange", label="Validation (2022)")
ax.axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2024-01-01"),
           alpha=0.10, color="seagreen",  label="Test (2023)")
ax.set_title("Daily Energy Generation — Valparaíso Region (2016–2023)", fontsize=14, weight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Total Generation (MWh)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("figures/01_generation_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.1 Normalization and Sequence Construction

LSTM networks are sensitive to input scale. Each region series is independently normalized to **[0, 1]** using Min-Max scaling, with the scaler fitted exclusively on the training partition (2016–2021) to prevent data leakage.

We then construct **sliding-window sequences of length 30 days**: each sample contains 30 consecutive daily observations as input and the next day as the target. This 30-day look-back window is consistent with the rolling input window described in Section 2.1 of the paper.

In [ ]:
SEQUENCE_LENGTH = 30   # look-back window in days (paper Section 2.1 and Appendix B.3)
BATCH_SIZE      = 32

# Min-Max normalization to [0, 1].
# NOTE: The full study (paper Appendix B.3) used z-score normalization (mean ± std
# of training split). This single-region toy case uses MinMaxScaler as an acceptable
# alternative — both approaches yield comparable LSTM accuracy on energy series.
# The scaler is fitted ONLY on the training partition (2016–2021) to prevent data leakage.
scaler = MinMaxScaler()

# Chronological splits — strict out-of-sample evaluation (no shuffling)
train_mask = df.index.year <= 2021
val_mask   = df.index.year == 2022
test_mask  = df.index.year == 2023

train_raw = df["TOTAL"].values[train_mask].reshape(-1, 1)
val_raw   = df["TOTAL"].values[val_mask  ].reshape(-1, 1)
test_raw  = df["TOTAL"].values[test_mask ].reshape(-1, 1)

scaler.fit(train_raw)
train_scaled = scaler.transform(train_raw)
val_scaled   = scaler.transform(val_raw)
test_scaled  = scaler.transform(test_raw)

def create_sequences(data, seq_len):
    """Convert a 1-D array into (X, y) sliding-window pairs."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
X_val,   y_val   = create_sequences(val_scaled,   SEQUENCE_LENGTH)
X_test,  y_test  = create_sequences(test_scaled,  SEQUENCE_LENGTH)

# Convert to PyTorch tensors
to_t = lambda a: torch.tensor(a, dtype=torch.float32)
X_train, y_train = to_t(X_train), to_t(y_train)
X_val,   y_val   = to_t(X_val),   to_t(y_val)
X_test,  y_test  = to_t(X_test),  to_t(y_test)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE)

print(f"Train sequences : {len(X_train):>5}  ({df.index[train_mask][SEQUENCE_LENGTH].date()} → {df.index[train_mask][-1].date()})")
print(f"Val sequences   : {len(X_val):>5}  ({df.index[val_mask ][SEQUENCE_LENGTH].date()} → {df.index[val_mask ][-1].date()})")
print(f"Test sequences  : {len(X_test):>5}  ({df.index[test_mask][SEQUENCE_LENGTH].date()} → {df.index[test_mask][-1].date()})")

## 3. Model Architecture

We use a stacked LSTM followed by a fully-connected output layer. The model takes a sequence of `SEQUENCE_LENGTH` (= 30) normalized daily generation values and outputs the prediction for the next day.

```
Input  (batch, 30, 1)
  └─► LSTM(hidden_size, num_layers)
        └─► last hidden state  (batch, hidden_size)
              └─► Dropout(p)
                    └─► Linear(hidden_size → 1) → Output (batch, 1)
```

**Dropout note:** `nn.LSTM` applies inter-layer dropout only when `num_layers > 1` — it silently ignores the `dropout` argument for single-layer stacks (PyTorch documented behaviour). To ensure regularisation is active regardless of depth, an explicit `nn.Dropout` layer is applied to the final hidden state before the linear projection. The number of layers, hidden size, dropout rate, and learning rate are all determined by the Optuna search in Section 4.

In [ ]:
class EnergyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super().__init__()
        # PyTorch's nn.LSTM silently ignores the `dropout` argument when num_layers == 1
        # (documented: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html).
        # We pass it through only for multi-layer stacks, and always apply an explicit
        # Dropout layer on the final hidden state so regularisation is active in all cases.
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :]))  # dropout → linear on last time-step

## 4. Hyperparameter Optimization

We use **Optuna** with the TPE (Tree-structured Parzen Estimator) sampler to search over four hyperparameters:

| Hyperparameter | Search range | Paper (Appendix B.4) |
|---|---|---|
| `hidden_size` (neurons per layer) | {32, …, 256} | Integer |
| `num_layers` | {1, 2, 3, 4} | Integer |
| `dropout` | [0.1, 0.5] | Continuous |
| `learning_rate` | [1×10⁻⁴, 1×10⁻²] (log) | Log-continuous |

Each trial trains for **40 epochs** (fast screening), evaluated on the 2022 validation MSE. The full study (paper Table 4) also benchmarks PSO and Genetic Algorithm under the same 50-trial budget per region.

The search block below is **commented out** (≈ 30 min on CPU). The hardcoded parameters correspond to the **Bayesian Optimization result for Valparaíso (R5, Table 4 of the paper)**:

| Parameter | Paper Table 4 (BO, R5) | This notebook |
|---|---|---|
| Neurons | 33 | 32 |
| Layers | 1 | 1 |
| Dropout | 0.394 | 0.398 |
| Learning rate | 4×10⁻⁴ | 1.588×10⁻³ |

Minor differences reflect stochastic search; both configurations share the same architecture class (1 layer, ~32 units, ~0.4 dropout).

In [6]:
INPUT_SIZE  = 1
OUTPUT_SIZE = 1

def objective(trial):
    hidden_size   = trial.suggest_int("hidden_size",  32, 256)
    num_layers    = trial.suggest_int("num_layers",   1, 4)
    dropout       = trial.suggest_float("dropout",    0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    _model     = EnergyLSTM(INPUT_SIZE, hidden_size, num_layers, OUTPUT_SIZE, dropout)
    _optimizer = torch.optim.Adam(_model.parameters(), lr=learning_rate)
    _criterion = nn.MSELoss()

    for _ in range(40):
        _model.train()
        for X_b, y_b in train_loader:
            _optimizer.zero_grad()
            _criterion(_model(X_b), y_b).backward()
            _optimizer.step()

    _model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            val_loss += _criterion(_model(X_b), y_b).item()
    return val_loss / len(val_loader)

# --- Uncomment to rerun the search (≈ 30 min on CPU for 50 trials) ---
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=50)
# best_params = study.best_params
# print("Best parameters found:", best_params)

# Best hyperparameters from pre-computed Optuna study:
best_params = {
    "hidden_size":   32,
    "num_layers":    1,
    "dropout":       0.398,
    "learning_rate": 0.001588,
}
print("Hyperparameters in use:", best_params)

Hyperparameters in use: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.398, 'learning_rate': 0.001588}


## 5. Model Training

The final model is trained for up to **200 epochs** using the optimal hyperparameters, with **early stopping** (patience = 20 epochs monitoring validation MSE). The best checkpoint is saved to `models/lstm_valparaiso.pt` and automatically restored after training.

In [ ]:
NUM_EPOCHS    = 200
PATIENCE      = 20   # early stopping patience (epochs without validation improvement)

model = EnergyLSTM(
    INPUT_SIZE,
    best_params["hidden_size"],
    best_params["num_layers"],
    OUTPUT_SIZE,
    best_params["dropout"],
)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=best_params["learning_rate"])

train_losses, val_losses = [], []
best_val_loss   = float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            val_loss += criterion(model(X_b), y_b).item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Early stopping & checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "models/lstm_valparaiso.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}  (best val loss: {best_val_loss:.6f})")
            break

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  Train: {train_loss:.6f}  Val: {val_loss:.6f}")

# Restore best weights
model.load_state_dict(torch.load("models/lstm_valparaiso.pt"))
print(f"\nBest model restored from models/lstm_valparaiso.pt  (val MSE = {best_val_loss:.6f})")

# Learning curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Training Loss",   color="steelblue")
ax.plot(val_losses,   label="Validation Loss", color="darkorange")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Learning Curves", fontsize=13, weight="bold")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("figures/02_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Model Evaluation

We evaluate the trained model on the **held-out 2023 test set**. Predictions are inverse-transformed to recover actual MWh values, and performance is reported using RMSE and MAPE.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test).numpy()
    y_true = y_test.numpy()

# Inverse-transform back to MWh
y_pred = scaler.inverse_transform(y_pred)
y_true = scaler.inverse_transform(y_true)

# Align dates: sequences start from position SEQUENCE_LENGTH within the 2023 block
test_dates = df.index[test_mask][SEQUENCE_LENGTH:]

results_df = pd.DataFrame({
    "DATE":       test_dates,
    "REAL":       y_true.flatten(),
    "PREDICTION": y_pred.flatten(),
})
results_df["ERROR"]          = results_df["PREDICTION"] - results_df["REAL"]
results_df["ABSOLUTE_ERROR"] = results_df["ERROR"].abs()
results_df["MAPE"]           = (results_df["ABSOLUTE_ERROR"] / results_df["REAL"]) * 100
results_df["MSE"]            = results_df["ERROR"] ** 2

rmse   = np.sqrt(results_df["MSE"].mean())
mae    = results_df["ABSOLUTE_ERROR"].mean()
mape   = results_df["MAPE"].mean()
bias   = results_df["ERROR"].mean()
d_stat = bias / results_df["ERROR"].std()

print(f"{'Metric':<30} {'Value':>15}")
print("-" * 47)
print(f"{'Test RMSE (MWh)':<30} {rmse:>15,.1f}")
print(f"{'Test MAE  (MWh)':<30} {mae:>15,.1f}")
print(f"{'Test MAPE (%)':<30} {mape:>15.2f}")
print(f"{'Mean Bias (MWh)':<30} {bias:>15,.1f}")
print(f"{'D-statistic':<30} {d_stat:>15.4f}")
print(f"{'Test periods':<30} {len(results_df):>15}  ({results_df['DATE'].min().date()} → {results_df['DATE'].max().date()})")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results_df["DATE"], results_df["REAL"],
        label="Ground Truth (2023)", linestyle="--", color="steelblue", linewidth=1.8)
ax.plot(results_df["DATE"], results_df["PREDICTION"],
        label="LSTM Prediction (2023)", color="darkorange", linewidth=1.8)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Energy Generation (MWh)", fontsize=12)
ax.set_title("LSTM Forecast vs Ground Truth — Test Set (2023)", fontsize=14, weight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("figures/03_forecast_vs_truth.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Full predictions table (DATE index, 304 rows: 2023-01-31 → 2023-11-30)
# Note: first prediction starts on the 31st day of 2023 because the 30-day
# look-back window consumes 2023-01-01 through 2023-01-30 as context.
display_df = results_df.set_index("DATE")[["REAL", "PREDICTION", "ABSOLUTE_ERROR", "MAPE"]].round(2)
print(f"Rows: {len(display_df)}  |  {display_df.index[0].date()} → {display_df.index[-1].date()}")
display_df

In [ ]:
# Save predictions to the data/ folder so Stage 2 (LP notebook) can load them directly
results_df.to_parquet("data/predictions_valparaiso_2023.parquet", index=False)
print("Predictions saved → data/predictions_valparaiso_2023.parquet")

## 7. Residual Analysis

We examine the forecast errors to assess model quality: ideally, residuals should be approximately normally distributed, centered at zero, and free of autocorrelation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram + KDE
sns.histplot(results_df["ABSOLUTE_ERROR"], kde=True, bins=30,
             color="royalblue", edgecolor="white", ax=axes[0])
axes[0].set_title("Absolute Error Distribution", weight="bold")
axes[0].set_xlabel("Absolute Error (MWh)")

# Normal Q-Q plot
stats.probplot(results_df["ABSOLUTE_ERROR"], dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot", weight="bold")

# Boxplot
sns.boxplot(x=results_df["ABSOLUTE_ERROR"], color="orange", ax=axes[2])
axes[2].set_title("Error Boxplot", weight="bold")
axes[2].set_xlabel("Absolute Error (MWh)")

plt.suptitle("Forecast Error — Distributional Analysis", fontsize=13, weight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figures/04_error_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter: predicted values vs residuals (checks for heteroskedasticity)
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(x=results_df["REAL"], y=results_df["ERROR"],
                color="seagreen", alpha=0.6, ax=ax)
ax.axhline(0, linestyle="--", color="gray", linewidth=1)
ax.set_title("Residuals vs Ground Truth", fontsize=13, weight="bold")
ax.set_xlabel("Ground Truth (MWh)")
ax.set_ylabel("Residual — Prediction minus Ground Truth (MWh)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/05_residuals_vs_truth.png", dpi=150, bbox_inches="tight")
plt.show()

# Normality tests on residuals
shapiro_stat, shapiro_p = stats.shapiro(results_df["ERROR"])
ks_stat,      ks_p      = stats.kstest(
    (results_df["ERROR"] - results_df["ERROR"].mean()) / results_df["ERROR"].std(),
    "norm"
)
print(f"Shapiro-Wilk  — W = {shapiro_stat:.4f},  p = {shapiro_p:.4f}")
print(f"KS (standard) — D = {ks_stat:.4f},  p = {ks_p:.4f}")
print(f"Overestimation  : {(results_df['PREDICTION'] > results_df['REAL']).mean()*100:.1f}%")
print(f"Underestimation : {(results_df['PREDICTION'] < results_df['REAL']).mean()*100:.1f}%")

In [ ]:
# Error over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(results_df["DATE"], results_df["ERROR"], color="crimson", linewidth=1.0)
ax.axhline(0, linestyle="--", color="black", linewidth=1)
ax.fill_between(results_df["DATE"], results_df["ERROR"], 0,
                where=results_df["ERROR"] > 0, alpha=0.3, color="tomato",    label="Over-forecast")
ax.fill_between(results_df["DATE"], results_df["ERROR"], 0,
                where=results_df["ERROR"] < 0, alpha=0.3, color="steelblue", label="Under-forecast")
ax.set_title("Forecast Error Over Time (2023)", fontsize=13, weight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Residual (MWh)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/06_error_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Autocorrelation and partial autocorrelation of the residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sm.graphics.tsa.plot_acf( results_df["ERROR"], ax=axes[0], lags=20)
sm.graphics.tsa.plot_pacf(results_df["ERROR"], ax=axes[1], lags=20)
axes[0].set_title("Autocorrelation Function (ACF)",          weight="bold")
axes[1].set_title("Partial Autocorrelation Function (PACF)", weight="bold")
plt.suptitle("Residual Autocorrelation Analysis", fontsize=13, weight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figures/07_residual_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Reproducibility Notes

When re-running this notebook, the reported metrics (RMSE, MAE, MAPE) may differ slightly from the values above for two reasons:

1. **Stochastic LSTM initialization.** Even with `torch.manual_seed(42)` and `np.random.seed(42)`, floating-point results can vary across CPUs, operating systems, and PyTorch versions due to non-deterministic CUDA kernels or changes in the internal random state ordering when using `DataLoader(shuffle=True)`. This is expected behavior documented in the [PyTorch reproducibility guide](https://pytorch.org/docs/stable/notes/randomness.html).

2. **Single-region toy case vs. full study.** The paper (Carrasco et al., 2026) trains one independent model per region across all 14 Chilean administrative regions. The hyperparameters reported in Table 4 of the paper are the best configuration found for each region. The hyperparameters hardcoded here (`hidden_size=32, num_layers=1, dropout=0.398, lr=0.001588`) correspond to the Bayesian Optimization result for Valparaíso; re-running the Optuna search (Section 4, commented-out block) may yield slightly different values due to the stochastic nature of the TPE sampler.

**Expected range:** Reproduced RMSE values within ± 5 % of the reported 3,626.8 MWh are consistent with normal stochastic variation. The LP dispatch outcomes in `Modelation_linear_programming.ipynb` are fully deterministic given the same prediction file — no random components are involved.